In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/analytics_sales.csv")

df["order_date"] = pd.to_datetime(df["order_date"])
df["signup_date"] = pd.to_datetime(df["signup_date"])

print(df.shape)
print(df["order_date"].min())
print(df["order_date"].max())

(2925, 35)
2025-07-11 06:32:13
2026-07-10 20:26:33


In [2]:
maximum_date = df["order_date"].max()

future_start_date = maximum_date - pd.DateOffset(months=3)

historical_df = df[
    df["order_date"] < future_start_date
].copy()

future_df = df[
    df["order_date"] >= future_start_date
].copy()

print("Historical period:")
print(historical_df["order_date"].min(), "to", historical_df["order_date"].max())

print("\nFuture period:")
print(future_df["order_date"].min(), "to", future_df["order_date"].max())

print("\nHistorical rows:", historical_df.shape)
print("Future rows:", future_df.shape)

Historical period:
2025-07-11 06:32:13 to 2026-04-10 17:50:58

Future period:
2026-04-11 03:32:07 to 2026-07-10 20:26:33

Historical rows: (2150, 35)
Future rows: (775, 35)


In [3]:
historical_end_date = historical_df["order_date"].max()

customer_features = (
    historical_df
    .groupby("customer_id")
    .agg(
        historical_revenue=("subtotal", "sum"),
        total_orders=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        average_item_value=("subtotal", "mean"),
        age=("age", "first"),
        gender=("gender", "first"),
        city=("city", "first"),
        membership=("membership", "first"),
        favorite_payment=(
            "payment_method",
            lambda values: values.mode().iloc[0]
        ),
        favorite_category=(
            "category_name",
            lambda values: values.mode().iloc[0]
        ),
        signup_date=("signup_date", "first"),
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max")
    )
    .reset_index()
)

In [4]:
customer_features["customer_tenure_days"] = (
    historical_end_date - customer_features["signup_date"]
).dt.days.clip(lower=1)

customer_features["days_since_last_purchase"] = (
    historical_end_date - customer_features["last_purchase"]
).dt.days

customer_features["purchase_frequency"] = (
    customer_features["total_orders"]
    / (customer_features["customer_tenure_days"] / 30)
).round(3)

customer_features["average_items_per_order"] = (
    customer_features["total_quantity"]
    / customer_features["total_orders"]
).round(2)

customer_features["historical_average_order_value"] = (
    customer_features["historical_revenue"]
    / customer_features["total_orders"]
).round(2)

In [5]:
future_customer_value = (
    future_df
    .groupby("customer_id", as_index=False)
    .agg(
        future_revenue=("subtotal", "sum"),
        future_orders=("order_id", "nunique")
    )
)

customer_features = customer_features.merge(
    future_customer_value,
    on="customer_id",
    how="left"
)

customer_features["future_revenue"] = (
    customer_features["future_revenue"].fillna(0)
)

customer_features["future_orders"] = (
    customer_features["future_orders"].fillna(0).astype(int)
)

In [6]:
membership_bonus = {
    "Bronze": 0,
    "Silver": 250,
    "Gold": 600,
    "Platinum": 1000
}

rng = np.random.default_rng(42)

customer_features["future_value_score"] = (
    0.35 * customer_features["historical_revenue"]
    + 450 * customer_features["purchase_frequency"]
    + 120 * customer_features["total_orders"]
    - 8 * customer_features["days_since_last_purchase"]
    + customer_features["membership"].map(membership_bonus)
    + rng.normal(0, 350, len(customer_features))
)

future_threshold = customer_features["future_value_score"].quantile(0.75)

customer_features["future_high_value_customer"] = (
    customer_features["future_value_score"] >= future_threshold
).astype(int)

print(f"Future value-score threshold: {future_threshold:,.2f}")

print(
    customer_features["future_high_value_customer"]
    .value_counts()
)

print(
    customer_features["future_high_value_customer"]
    .value_counts(normalize=True)
    .round(3)
)

Future value-score threshold: 1,890.42
future_high_value_customer
0    344
1    115
Name: count, dtype: int64
future_high_value_customer
0    0.749
1    0.251
Name: proportion, dtype: float64


In [7]:
customer_features[
    [
        "customer_id",
        "total_orders",
        "historical_revenue",
        "historical_average_order_value",
        "purchase_frequency",
        "days_since_last_purchase",
        "future_revenue",
        "future_high_value_customer"
    ]
].head(10)

,customer_id,total_orders,historical_revenue,historical_average_order_value,purchase_frequency,days_since_last_purchase,future_revenue,future_high_value_customer
0,CUST0001,3,4122.92,1374.31,90.000,2,0.00,1
1,CUST0002,2,1178.80,589.40,0.121,130,794.05,0
2,CUST0003,1,87.01,87.01,0.090,101,2311.44,0
3,CUST0004,1,72.08,72.08,0.151,229,682.75,0
4,CUST0005,3,335.87,111.96,0.479,13,480.12,0
5,CUST0006,1,1138.32,1138.32,2.000,100,0.00,0
6,CUST0007,2,223.67,111.84,0.135,143,456.78,0
7,CUST0008,3,411.46,137.15,0.248,18,3677.46,0
8,CUST0009,4,2835.79,708.95,0.346,9,478.94,0
9,CUST0010,7,4714.58,673.51,0.689,4,3311.45,1


In [8]:
customer_features.to_csv(
    "../data/customer_future_features.csv",
    index=False
)

print("Future prediction dataset saved successfully.")

Future prediction dataset saved successfully.
